In [53]:
import sqlite3
from uuid import uuid4
from datetime import datetime

In [5]:
con = sqlite3.connect("../volumes/data/app.db")
cur = con.cursor()

In [9]:
try:
    cur.execute("CREATE TABLE test (id INTEGER PRIMARY KEY, name TEXT)")
    con.commit()
    print("Table created successfully.")
except sqlite3.OperationalError:
    print("Table already exists.")

Table already exists.


In [13]:
res = cur.execute("SELECT name FROM sqlite_master WHERE type='table';")
print(res.fetchall())

[('test',), ('users',), ('sessions',), ('responses',)]


In [35]:
def create_tables() -> None:
    tables = {
        "users": {"userID": "TEXT PRIMARY KEY", "username": "TEXT"},
        "sessions": {"sessionID": "TEXT PRIMARY KEY", "userID": "TEXT", "name": "TEXT", "createdAt": "TEXT", "updatedAt": "TEXT"},
        "responses": {"responseID": "TEXT PRIMARY KEY", "sessionID": "TEXT", "prompt": "TEXT", "response": "TEXT", "timestamp": "TEXT"}
    }
    for table, columns in tables.items():
        columns_def = ", ".join([f"{col} {dtype}" for col, dtype in columns.items()])
        try:
            cur.execute(f"CREATE TABLE {table} ({columns_def})")
            con.commit()
            print(f"Table \'{table}\' created successfully.")
        except sqlite3.OperationalError:
            print(f"Table \'{table}\' already exists.")

In [36]:
create_tables()

Table 'users' already exists.
Table 'sessions' already exists.
Table 'responses' already exists.


In [32]:
def new_user(userID: str, username: str) -> None:
    try:
        cur.execute("INSERT INTO users (userID, username) VALUES (?, ?)", (userID, username))
        con.commit()
        print(f"User \'{username}\' with ID \'{userID}\' added successfully.")
    except sqlite3.IntegrityError:
        print(f"User \'{username}\' with ID \'{userID}\' already exists.")

In [40]:
user_id = 'f6885c0d-b788-470c-b5f3-0314d70e8a42'
session_id = 'a938301c-1c59-4379-a5ed-c7ad0eac2c2a'

In [34]:
new_user(user_id, "testuser")

User 'testuser' with ID 'f6885c0d-b788-470c-b5f3-0314d70e8a42' already exists.


In [17]:
con.execute("SELECT * FROM users").fetchall()

[('f6885c0d-b788-470c-b5f3-0314d70e8a42', 'testuser')]

In [ ]:
def new_session(session_id: str, user_id: str, name: str, date_time: str|None = None) -> None:
    now = datetime.now().isoformat() if date_time is None else date_time
    try:
        cur.execute("INSERT INTO sessions (sessionID, userID, name, createdAt, updatedAt) VALUES (?, ?, ?, ?, ?)", (session_id, user_id, name, now, now))
        con.commit()
        print(f"Session \'{name}\' with ID \'{session_id}\' added successfully.")
    except sqlite3.IntegrityError:
        print(f"Session \'{name}\' with ID \'{session_id}\' already exists.")

In [77]:
new_session(session_id, user_id, "Test Session")

Session 'Test Session' with ID 'a938301c-1c59-4379-a5ed-c7ad0eac2c2a' added successfully.


In [80]:
con.execute("SELECT * FROM sessions").fetchall()

[('a938301c-1c59-4379-a5ed-c7ad0eac2c2a',
  'f6885c0d-b788-470c-b5f3-0314d70e8a42',
  'Test Session',
  '2025-11-23T13:27:25.980511Z',
  '2025-11-23T13:27:28.326953Z')]

In [ ]:
def new_response(session_id: str, response_id: str, prompt: str, response: str) -> None:
    now = datetime.now().isoformat()
    new_session(session_id, user_id, "Test Session", now)

    try:
        cur.execute("UPDATE sessions SET updatedAt = ? WHERE sessionID = ?", (now, session_id))
        cur.execute("INSERT INTO responses (responseID, sessionID, prompt, response, timestamp) VALUES (?, ?, ?, ?, ?)", (response_id, session_id, prompt, response, now))
        con.commit()
        print(f"Response with ID \'{response_id}\' added successfully.")
    except sqlite3.IntegrityError:
        print(f"Response with ID \'{response_id}\' already exists.")

In [89]:
new_response(session_id, str(uuid4()), "Hello, how are you?", "I'm fine, thank you!")

Session 'Test Session' with ID 'a938301c-1c59-4379-a5ed-c7ad0eac2c2a' already exists.
Response with ID '3e674274-698f-4525-831e-8a318b8878bb' added successfully.


In [90]:
con.execute("SELECT * FROM responses ORDER BY timestamp DESC").fetchall()

[('3e674274-698f-4525-831e-8a318b8878bb',
  'a938301c-1c59-4379-a5ed-c7ad0eac2c2a',
  'Hello, how are you?',
  "I'm fine, thank you!",
  '2025-11-23T13:33:43.207404'),
 ('81540d92-cab8-4494-a1e0-cc566dc1005f',
  'a938301c-1c59-4379-a5ed-c7ad0eac2c2a',
  'Hello, how are you?',
  "I'm fine, thank you!",
  '2025-11-23T13:33:28.964257'),
 ('4b168cb9-3d1c-4456-90ab-1321326a9497',
  'a938301c-1c59-4379-a5ed-c7ad0eac2c2a',
  'Hello, how are you?',
  "I'm fine, thank you!",
  '2025-11-23T13:27:48.727315Z'),
 ('45e79448-750a-401c-9e98-71251d2918cf',
  'a938301c-1c59-4379-a5ed-c7ad0eac2c2a',
  'Hello, how are you?',
  "I'm fine, thank you!",
  '2025-11-23T13:27:47.863246Z'),
 ('fa4cb8dd-a6c4-4f00-b701-e1655b7c9575',
  'a938301c-1c59-4379-a5ed-c7ad0eac2c2a',
  'Hello, how are you?',
  "I'm fine, thank you!",
  '2025-11-23T13:27:45.767209Z'),
 ('469a2fc6-a1a2-43de-ae09-d23085028f08',
  'a938301c-1c59-4379-a5ed-c7ad0eac2c2a',
  'Hello, how are you?',
  "I'm fine, thank you!",
  '2025-11-23T13:27:28